# Topic Modeling Validation Notebook

This notebook evaluates an LDA topic model on Yelp reviews using intrinsic and extrinsic metrics: Perplexity, Coherence (c_v and UMass), Topic Diversity, Topic Stability, and Downstream Classification Performance.

In [ ]:
# %pip install -U scikit-learn gensim pandas numpy nltk

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alche\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
df = pd.read_parquet("../..//datasets/cleaned_datasets/demographic_infused_philly_c.parquet")
df = df[df['clean_text'].notna()]
texts = df['clean_text'].tolist()

In [ ]:
tokenized = [word_tokenize(doc.lower()) for doc in texts]
dictionary = Dictionary(tokenized)
corpus = [dictionary.doc2bow(doc) for doc in tokenized]

In [8]:
# TF-IDF vectorization and LDA fitting
vectorizer = TfidfVectorizer(stop_words='english',
                             max_df=0.95,
                             min_df=10)
tfidf = vectorizer.fit_transform(texts)
lda = LatentDirichletAllocation(n_components=10,
                                random_state=42,
                                evaluate_every=1)
lda.fit(tfidf)
doc_topics = lda.transform(tfidf)

In [9]:
# Extract top words per topic
feature_names = vectorizer.get_feature_names_out()
top_n = 10
topics = []
for idx, comp in enumerate(lda.components_):
    top_features = [feature_names[i] for i in comp.argsort()[-top_n:][::-1]]
    topics.append(top_features)
    print(f"Topic {idx}: {top_features}")

Topic 0: ['taco', 'food', 'good', 'chicken', 'delicious', 'place', 'thai', 'great', 'pho', 'ordered']
Topic 1: ['wedding', 'service', 'day', 'customer', 'card', 'time', 'email', 'package', 'flower', 'order']
Topic 2: ['pizza', 'cheesesteak', 'sandwich', 'good', 'place', 'cheese', 'philly', 'best', 'great', 'food']
Topic 3: ['order', 'food', 'time', 'ordered', 'service', 'dont', 'minute', 'place', 'customer', 'delivery']
Topic 4: ['place', 'time', 'people', 'table', 'room', 'like', 'food', 'store', 'staff', 'drink']
Topic 5: ['good', 'delicious', 'great', 'food', 'got', 'cheese', 'place', 'coffee', 'sandwich', 'ordered']
Topic 6: ['food', 'good', 'great', 'place', 'sushi', 'delicious', 'restaurant', 'service', 'roll', 'menu']
Topic 7: ['car', 'work', 'time', 'service', 'job', 'recommend', 'company', 'day', 'professional', 'great']
Topic 8: ['great', 'food', 'service', 'drink', 'amazing', 'place', 'beer', 'staff', 'atmosphere', 'friendly']
Topic 9: ['nail', 'hair', 'salon', 'time', 'appo

In [10]:
# 1. Perplexity (lower is better)
perplexity = lda.perplexity(tfidf)
print('Perplexity:', perplexity)

Perplexity: 6380.226739166576


In [11]:
# 2. Topic Coherence
cm_cv = CoherenceModel(topics=topics,
                    texts=tokenized,
                    dictionary=dictionary,
                    coherence='c_v')
print('Coherence (c_v):', cm_cv.get_coherence())
cm_umass = CoherenceModel(topics=topics,
                        corpus=corpus,
                        dictionary=dictionary,
                        coherence='u_mass')
print('Coherence (u_mass):', cm_umass.get_coherence())

Coherence (c_v): 0.4977584077181209
Coherence (u_mass): -2.071662056521631


In [12]:
# 3. Topic Diversity
all_words = [w for topic in topics for w in topic]
diversity = len(set(all_words)) / len(all_words)
print('Topic Diversity:', diversity)

Topic Diversity: 0.59


In [13]:
# 4. Topic Stability (Jaccard similarity across two LDA runs)
def jaccard(a, b):
    return len(set(a)&set(b)) / len(set(a)|set(b))
lda2 = LatentDirichletAllocation(n_components=10, random_state=0)
lda2.fit(tfidf)
topics2 = [[feature_names[i] for i in comp.argsort()[-top_n:][::-1]]
           for comp in lda2.components_]
sims = [jaccard(t1, t2) for t1, t2 in zip(topics, topics2)]
print('Topic Stability (mean Jaccard):', np.mean(sims))

Topic Stability (mean Jaccard): 0.2193211157607442


In [14]:
# 5. Downstream Classification
# Binary label: high rating >=4
y = (df['stars_rev'] >= 4).astype(int)
X_train, X_test, y_train, y_test = train_test_split(doc_topics, y, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
y_pred = clf.predict(X_test)
print('Downstream Accuracy:', accuracy_score(y_test, y_pred))

Downstream Accuracy: 0.7947418285172904
